# Домашнее задание №3

# Задание

[Ссылка](https://tn-gvu.mckx.ru/c/c4YcAACAurcBAEAA/sE43BQ/ZYcLPOYT_IPW6LK3/?u=https%3A%2F%2Fcontest.yandex.ru%2Fcontest%2F67637%2Fenter%2F%3Futm_source%3Dmindbox%26utm_medium%3Demail%26utm_campaign%3Dtraining6%26utm_content%3Ddigest%26utm_term%3D06.11.2024) на контест с заданием.

Необходимо обучить модель перевода с языка зетан на английский.

***Данные:*** https://disk.yandex.ru/d/u8mmcUBn64p_Nw

***Формат решения:*** json-lines в формате, аналогичной валидации, с переводами исходных текстов в тестовой выборке

***Метрика качества:*** BLEU между переводами и английскими референсами на закрытом тесте

## Загрузка данных

In [ ]:
# ! wget https://disk.yandex.ru/d/u8mmcUBn64p_Nw

--2024-11-13 18:29:50--  https://disk.yandex.ru/d/u8mmcUBn64p_Nw
Resolving disk.yandex.ru (disk.yandex.ru)... 87.250.250.50, 2a02:6b8::2:50
Connecting to disk.yandex.ru (disk.yandex.ru)|87.250.250.50|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 68985 (67K) [text/html]
Saving to: ‘u8mmcUBn64p_Nw’

u8mmcUBn64p_Nw      100%[===================>]  67.37K   322KB/s    in 0.2s    

2024-11-13 18:29:51 (322 KB/s) - ‘u8mmcUBn64p_Nw’ saved [68985/68985]



In [13]:
from datasets import load_dataset

data_files = {
    "train": "train",
    "validation": "val",
    "test": "test_no_reference"
}

dataset = load_dataset("json", data_files=data_files)

dataset

DatasetDict({
    train: Dataset({
        features: ['dst', 'src'],
        num_rows: 300000
    })
    test: Dataset({
        features: ['dst', 'src'],
        num_rows: 1000
    })
    validation: Dataset({
        features: ['dst', 'src'],
        num_rows: 500
    })
})

Посмотрим на один пример данных. Он сотсоит из словаря содержащего пару `src` - исходная фраза на языке Зеттан, `dst` - целевая фраза на английском языке.

In [21]:
dataset['train'][4]

{'dst': "He's talking about a few right here in Lisbon, mostly Jews who have escaped the Germans, that's all.",
 'src': '◈◠ ◧▱◠▦ ◀◫◓ ▨◠◉ ◂▱◠▽◈◠▦ ◀◠▷◞◪◈◗◳◧◓■ ◉◧◐▾▦▱◨◐▾ ○▱◎◠▦▱◠◓◈◠▦ ▨◠◉◠▦ ▽◠▷◨◈◫▱▴◓▵'}

Немного узнать о этом чудном языке не помешает. Посмотрим сколько уникальных символов в этом языке:

In [22]:
# Функция для подсчета уникальных символов
def count_unique_characters(dataset, feature_name):
    unique_characters = set()
    for split in dataset.keys():
        for example in dataset[split][feature_name]:
            unique_characters.update(example)
    return len(unique_characters), unique_characters

# Подсчет уникальных символов во всех частях датасета
num_unique_chars, unique_chars = count_unique_characters(dataset, "src")

print(f"Количество уникальных символов: {num_unique_chars}")
print(f"Уникальные символы: {unique_chars}")

Количество уникальных символов: 182
Уникальные символы: {'◥', '◜', '°', '6', '◓', '▷', 'û', '9', 'ë', '¿', 'þ', 'Â', '8', '▲', '▵', '□', '◬', '4', '’', '▾', '◀', '▱', 'ï', '2', '£', 'ñ', ' ', '•', '◘', '◞', '▽', 'ú', '◤', '¡', '◫', '~', 'º', '◎', '◄', '▿', '◧', '◭', '▤', '♪', 'ý', '◝', '\u200b', '‚', '5', 'ν', '▶', '´', 'é', '\x9e', '◮', 'ª', '[', '”', '◇', '○', '■', 'ß', 'ó', '►', '`', '▣', 'å', ':', 'ţ', '\x9d', '◗', '½', '▭', '\x99', '◒', '“', '^', '"', 'É', '◁', '?', ';', 'ø', '◠', '▨', '0', '\xa0', '◩', 'â', 'í', '◙', '◚', 'Ý', '◆', '▯', '\\', '▬', '◍', 'è', '¤', '™', '▪', 'ä', ')', '▮', '◂', '◌', '◣', '◡', '▼', '%', '◱', '▧', 'Ä', '1', '7', 'î', '◊', '◪', 'á', '◳', '◐', '◛', '◖', '●', 'Þ', '♫', '△', 'ο', '▥', "'", '▢', '◈', '◢', 'ι', 'ă', '/', '◕', ']', '◅', '*', '–', '_', '=', '@', '◃', '◑', '!', '▹', '¶', '{', '$', '3', '◨', 'ð', '◯', 'đ', '─', '◲', '▸', '▴', '▻', '—', '#', '+', '▩', '◦', '▦', '▫', '▰', 'ô', '§', '‘', 'Î', '&', '◔', '◰', '(', '}', '◉', '◟', '\u202d'}


## Токенизация данных

Для начала выберем модель, которую будет использовать для дообучения. Попробуем использовать модель :

In [25]:
model_checkpoint = "Helsinki-NLP/opus-mt-ko-en"

In [28]:
! pip install sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 5.0 MB/s eta 0:00:00a 0:00:01


Попробуем обучить новый токнизатор для данного языка:

In [29]:
from transformers import MarianTokenizer, MarianMTModel
from tokenizers import ByteLevelBPETokenizer
from transformers import AutoTokenizer



tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
tokenizer.is_fast

# tokenizer.train()

ValueError: This tokenizer cannot be instantiated. Please make sure you have `sentencepiece` installed in order to use this tokenizer.

Количество уникальных символов: 182
Уникальные символы: {'◥', '◜', '°', '6', '◓', '▷', 'û', '9', 'ë', '¿', 'þ', 'Â', '8', '▲', '▵', '□', '◬', '4', '’', '▾', '◀', '▱', 'ï', '2', '£', 'ñ', ' ', '•', '◘', '◞', '▽', 'ú', '◤', '¡', '◫', '~', 'º', '◎', '◄', '▿', '◧', '◭', '▤', '♪', 'ý', '◝', '\u200b', '‚', '5', 'ν', '▶', '´', 'é', '\x9e', '◮', 'ª', '[', '”', '◇', '○', '■', 'ß', 'ó', '►', '`', '▣', 'å', ':', 'ţ', '\x9d', '◗', '½', '▭', '\x99', '◒', '“', '^', '"', 'É', '◁', '?', ';', 'ø', '◠', '▨', '0', '\xa0', '◩', 'â', 'í', '◙', '◚', 'Ý', '◆', '▯', '\\', '▬', '◍', 'è', '¤', '™', '▪', 'ä', ')', '▮', '◂', '◌', '◣', '◡', '▼', '%', '◱', '▧', 'Ä', '1', '7', 'î', '◊', '◪', 'á', '◳', '◐', '◛', '◖', '●', 'Þ', '♫', '△', 'ο', '▥', "'", '▢', '◈', '◢', 'ι', 'ă', '/', '◕', ']', '◅', '*', '–', '_', '=', '@', '◃', '◑', '!', '▹', '¶', '{', '$', '3', '◨', 'ð', '◯', 'đ', '─', '◲', '▸', '▴', '▻', '—', '#', '+', '▩', '◦', '▦', '▫', '▰', 'ô', '§', '‘', 'Î', '&', '◔', '◰', '(', '}', '◉', '◟', '\u202d'}
